# Cox PH — PCI × Tumour Type Interaction
## Does the prognostic effect of PCI differ by cancer type?

**Hypothesis:** PCI may be more (or less) prognostic in certain tumour types.  
A significant interaction would mean that reporting a single PCI HR across all cancers is misleading.

**Approach:**
1. Test the PCI × Tumour interaction term with an LRT (interaction model vs main-effects model)
2. Visualise per-tumour-type PCI HRs from the interaction model
3. Fit stratified subgroup Cox models (one per tumour type) as a sensitivity check


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
from scipy.stats import chi2

import os, sys
library_path = Path(os.path.abspath('../src'))
if str(library_path) not in sys.path:
    sys.path.append(str(library_path))

DATA_PATH  = library_path.parent / "data"
PLOTS_PATH = library_path.parent / "plots"


In [ ]:
# ── Load & prepare data ───────────────────────────────────────────────────────
cols = ["event", "months", "Sex", "Age", "Tumor", "sPCI", "pPCI", "CC"]
df_raw = pd.read_csv(DATA_PATH / "GPT_processed_survival_data.csv", usecols=cols)

df_raw["months_round"] = df_raw["months"].round().astype(int)
t_cutoff = 57
df_raw["months_trunc"] = df_raw["months_round"].clip(upper=t_cutoff)
df_raw["event_trunc"]  = ((df_raw["event"] == 1) & (df_raw["months_round"] <= t_cutoff)).astype(int)
df_raw["Sex_01"]       = df_raw["Sex"] - 1  # 0 = female, 1 = male

# Tumour label map (type → readable name for plots)
TUMOR_LABELS = {
    1: "Type 1", 2: "Type 2", 3: "Type 3", 4: "Type 4",
    5: "Type 5", 6: "Type 6", 7: "Type 7", 8: "Type 8",
}

tumor_counts = df_raw["Tumor"].value_counts().sort_index()
print("Tumour type N:")
display(tumor_counts.rename("n").to_frame().T)
print(f"\nTotal N={len(df_raw)}, Events={df_raw['event_trunc'].sum()}")


In [ ]:

# ── Build interaction DataFrame ────────────────────────────────────────────────
# Design:
#   • Tumor is kept as a plain column → used as stratum, not as a covariate
#   • Interaction terms: PCI × T_k for every k (T_k dummies are NOT included in the model)
#   • Each PCI×T_k term equals PCI within stratum k and 0 elsewhere.
#     Stratifying by Tumor absorbs baseline differences; each γ_k is the
#     per-type PCI slope estimated entirely from stratum k's data.

def build_interaction_df(df_raw: pd.DataFrame, pci_col: str):
    """
    Returns (df_model, tumor_dummy_cols, pci_x_tumor_cols) where df_model contains:
      Age, CC, Tumor (for stratification), PCI×T_1 … PCI×T_k
    Tumor is kept as a column for strata=["Tumor"]; no dummy columns are included.
    """
    d = (df_raw[["event_trunc", "months_trunc", "Age", "Tumor", "CC", pci_col]]
         .dropna(subset=[pci_col])
         .reset_index(drop=True)
         .copy())
    d.rename(columns={"event_trunc": "event", "months_trunc": "months"}, inplace=True)

    # Build T_k dummies only to compute interaction terms — not added to model
    dummies = pd.get_dummies(d["Tumor"], prefix="T", drop_first=False, dtype=float)
    tumor_dummy_cols = sorted(dummies.columns.tolist())   # T_1, T_2, …

    # Interaction terms: PCI × T_k for each k
    pci_x_tumor_cols = []
    for tc in tumor_dummy_cols:
        col_name = f"{pci_col}_x_{tc}"
        dummies[col_name] = d[pci_col] * dummies[tc]
        pci_x_tumor_cols.append(col_name)

    # Keep Tumor for stratification; only append interaction columns (not raw dummies)
    d = pd.concat([d, dummies[pci_x_tumor_cols]], axis=1)
    return d, tumor_dummy_cols, pci_x_tumor_cols

df_S, tumor_cols_S, int_cols_S = build_interaction_df(df_raw, "sPCI")
df_P, tumor_cols_P, int_cols_P = build_interaction_df(df_raw, "pPCI")

print("Columns in Model A (sPCI):")
print("  Interaction terms  :", int_cols_S)
print(f"\nN = {len(df_S)}, Events = {int(df_S['event'].sum())}")
print("\nEvents per Tumour type:")
display(df_S.groupby("Tumor")["event"].agg(["sum", "count"]).rename(columns={"sum": "events", "count": "n"}))


In [ ]:

# ── Fit Cox models with PCI × Tumor interaction ───────────────────────────────
# Model: Age + CC + PCI×T_1 + … + PCI×T_k, stratified by Tumor.
# Stratifying by Tumor absorbs each type's baseline hazard entirely.
# Within stratum k, PCI×T_k = PCI and all other interaction terms = 0, so
# γ_k is estimated purely from that stratum's data — no collinearity.
# No ridge penalty needed.

PENALIZER = 0.0

def fit_interaction_model(df, pci_x_tumor_cols, label, penalizer=PENALIZER):
    fit_cols = ["Age", "CC"] + pci_x_tumor_cols + ["event", "months", "Tumor"]
    cph = CoxPHFitter(penalizer=penalizer)
    cph.fit(df[fit_cols], duration_col="months", event_col="event", strata=["Tumor"])
    print(f"\n{'='*62}")
    print(f" {label}")
    print(f"{'='*62}")
    cph.print_summary(decimals=3)
    return cph

cph_S = fit_interaction_model(df_S, int_cols_S, "MODEL A — sPCI × Tumour (stratified by Tumour)")
cph_P = fit_interaction_model(df_P, int_cols_P, "MODEL B — pPCI × Tumour (stratified by Tumour)")


In [ ]:
# ── Per-tumour-type PCI HR (directly from PCI×T_k coefficients) ───────────────
# Because each PCI×T_k term is the log-HR for PCI within tumour type k,
# exp(β_{PCI×T_k}) is the HR per unit PCI for patients with that tumour type.
# (No delta-method needed: each interaction coeff is already the full per-type slope.)

def extract_per_tumor_hr(cph, pci_x_tumor_cols, tumor_labels):
    rows = []
    for col in pci_x_tumor_cols:
        t_num = int(col.split("_")[-1])
        b     = cph.params_[col]
        se    = np.sqrt(cph.variance_matrix_.loc[col, col])
        rows.append({
            "Tumor"  : t_num,
            "Label"  : tumor_labels.get(t_num, f"T{t_num}"),
            "log_HR" : b,
            "HR"     : np.exp(b),
            "CI_lo"  : np.exp(b - 1.96 * se),
            "CI_hi"  : np.exp(b + 1.96 * se),
            "p"      : cph.summary.loc[col, "p"],
        })
    return pd.DataFrame(rows).sort_values("Tumor").reset_index(drop=True)

hr_S = extract_per_tumor_hr(cph_S, int_cols_S, TUMOR_LABELS)
hr_P = extract_per_tumor_hr(cph_P, int_cols_P, TUMOR_LABELS)

print("sPCI HR per tumour type:")
display(hr_S[["Label", "HR", "CI_lo", "CI_hi", "p"]].round(3))

print("\npPCI HR per tumour type:")
display(hr_P[["Label", "HR", "CI_lo", "CI_hi", "p"]].round(3))


In [ ]:
# ── Forest plot: PCI HR per tumour type (sPCI vs pPCI side-by-side) ───────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, hr_df, pci_label, color in [
    (axes[0], hr_S, "sPCI", "steelblue"),
    (axes[1], hr_P, "pPCI", "darkorange"),
]:
    ys = range(len(hr_df))
    for i, row in hr_df.iterrows():
        sig   = row["p"] < 0.05
        msize = 9 if sig else 6
        lw    = 2.0 if sig else 1.2
        ec    = color if sig else "gray"
        ax.errorbar(
            row["HR"], i,
            xerr=[[row["HR"] - row["CI_lo"]], [row["CI_hi"] - row["HR"]]],
            fmt="o", color=ec, capsize=4, markersize=msize, linewidth=lw,
        )

    ax.set_yticks(list(ys))
    ax.set_yticklabels(hr_df["Label"].tolist(), fontsize=9)
    ax.axvline(1, color="black", lw=1, ls="--", alpha=0.5)
    ax.set_xscale("log")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:g}"))
    ax.set_xlabel("HR per unit PCI (95% CI, log scale)", fontsize=9)
    ax.set_title(f"{pci_label} × Tumour type\n(filled = p < 0.05)", fontsize=11)
    ax.grid(axis="x", alpha=0.25)

plt.suptitle("Per-Tumour-Type PCI Hazard Ratios (Interaction Model)", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(PLOTS_PATH / "cox_interaction_per_tumor_HR.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:

# ── Global interaction test: main-effects vs interaction model (LRT) ──────────
# Main-effects model: Age, CC, PCI — stratified by Tumor, no interaction.
# Both models share the same stratification, so the LRT is valid.
# df = k - 1: interaction model has k PCI slopes vs 1 global slope.

def fit_main_effects(df, pci_col, penalizer=PENALIZER):
    """Main-effects Cox: Age, CC, PCI — stratified by Tumor, no interaction."""
    fit_cols = ["Age", "CC", pci_col, "event", "months", "Tumor"]
    cph = CoxPHFitter(penalizer=penalizer)
    cph.fit(df[fit_cols], duration_col="months", event_col="event", strata=["Tumor"])
    return cph

cph_S_main = fit_main_effects(df_S, "sPCI")
cph_P_main = fit_main_effects(df_P, "pPCI")

for label, cph_main, cph_int, int_cols in [
    ("sPCI", cph_S_main, cph_S, int_cols_S),
    ("pPCI", cph_P_main, cph_P, int_cols_P),
]:
    lrt  = 2 * (cph_int.log_likelihood_ - cph_main.log_likelihood_)
    dof  = len(int_cols) - 1   # k per-type slopes vs 1 global slope → df = k-1
    p    = chi2.sf(lrt, df=dof)
    dc   = cph_int.concordance_index_ - cph_main.concordance_index_
    print(f"\n{label}:")
    print(f"  Main-effects  C-index = {cph_main.concordance_index_:.4f}")
    print(f"  Interaction   C-index = {cph_int.concordance_index_:.4f}  (ΔC = {dc:+.4f})")
    print(f"  LRT χ²({dof})   = {lrt:.3f},  p = {p:.4f}")
    if p < 0.05:
        print(f"  → Significant: PCI effect on OS differs across tumour types.")
    else:
        print(f"  → Not significant: no evidence that PCI effect varies by tumour type.")
